In [ ]:
!pip install opensmile
!python -m spacy download es_core_news_sm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 996.0/996.0 kB 48.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.3/70.3 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.9/41.9 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 150.9/150.9 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 51.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.4/138.4 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.8/324.8 kB 29.5 MB/s eta 0:00:00
Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/pip/_internal/cli/base_command.py", line 179, in exc_logging_wrapper
    status = run_func(*args)
             ^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/pip/_internal/cli/req_command.py", line 67, in wrapper
    return func(self, options, args)
           ^^^^

# Extracción de métricas acústicas y lingüísticas

Este notebook está basado en el enfoque descrito en el artículo [PMC9056005](https://pmc.ncbi.nlm.nih.gov/articles/PMC9056005/), adaptado para analizar chunks de audio y sus correspondientes transcripciones en un contexto de clasificación de afasia.

## Métricas Extraídas

Las siguientes métricas se generan a partir de los audios y transcripciones procesados:

### **Métricas Acústicas (OpenSMILE eGeMAPS)**
- `F0semitoneFrom27.5Hz_sma3nz_amean`
- `F0semitoneFrom27.5Hz_sma3nz_stddevNorm`
- `F0semitoneFrom27.5Hz_sma3nz_percentile20.0`
- `F0semitoneFrom27.5Hz_sma3nz_percentile50.0`
- `F0semitoneFrom27.5Hz_sma3nz_percentile80.0`
- `F0semitoneFrom27.5Hz_sma3nz_pctlrange0-2`
- `F0semitoneFrom27.5Hz_sma3nz_meanRisingSlope`
- `F0semitoneFrom27.5Hz_sma3nz_stddevRisingSlope`
- `F0semitoneFrom27.5Hz_sma3nz_meanFallingSlope`
- `F0semitoneFrom27.5Hz_sma3nz_stddevFallingSlope`
- `loudness_sma3_amean`
- `loudness_sma3_stddevNorm`
- `spectralFlux_sma3_amean`
- `spectralFlux_sma3_stddevNorm`
- `mfcc1_sma3_amean` a `mfcc4_sma3_stddevNorm`
- `jitterLocal_sma3nz_amean`
- `jitterLocal_sma3nz_stddevNorm`
- `shimmerLocaldB_sma3nz_amean`
- `shimmerLocaldB_sma3nz_stddevNorm`
- `HNRdBACF_sma3nz_amean`
- `HNRdBACF_sma3nz_stddevNorm`
- `alphaRatioV_sma3nz_amean`
- `alphaRatioV_sma3nz_stddevNorm`
- `hammarbergIndexV_sma3nz_amean`
- `hammarbergIndexV_sma3nz_stddevNorm`

### **Métricas Lingüísticas**
- Número total de palabras (`num_palabras`).
- Número de palabras únicas (`num_palabras_unicas`).
- Diversidad léxica (`diversidad_lexica`): proporción de palabras únicas respecto al total.
- Embeddings de BERT para las transcripciones.

### **Métricas Derivadas (Librosa)**
- `mfcc1_mean` a `mfcc13_mean`: Medias de los coeficientes MFCC.
- `mfcc1_stddev` a `mfcc13_stddev`: Desviaciones estándar de los coeficientes MFCC.

Estas métricas se combinan con las columnas originales del dataset para formar un único conjunto de datos listo para la modelización.

In [ ]:
import librosa
import json
import numpy as np
import pandas as pd
from opensmile import Smile,FeatureSet, FeatureLevel
from transformers import BertTokenizer, BertModel
import spacy

ModuleNotFoundError: No module named 'opensmile'

In [ ]:
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

path_data = '/content/drive/MyDrive/tesis_monica/afasia/data/'

In [ ]:
# Inicialización de librerías necesarias
nlp = spacy.load("es_core_news_sm")  # Para métricas lingüísticas
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")  # BERT embeddings
model = BertModel.from_pretrained("bert-base-uncased")

# Inicializar OpenSMILE para características acústicas específicas
smile_egemaps = Smile(feature_set=FeatureSet.eGeMAPSv02, feature_level=FeatureLevel.Functionals)

# Función para extraer características acústicas con librosa
def extraer_mfcc(y, sr):
    mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
    delta_mfcc = librosa.feature.delta(mfccs)
    delta2_mfcc = librosa.feature.delta(mfccs, order=2)

    # Calcular estadísticas resumen
    features = {
        f"mfcc{i+1}_mean": np.mean(mfcc) for i, mfcc in enumerate(mfccs)
    }
    features.update({
        f"mfcc{i+1}_stddev": np.std(mfcc) for i, mfcc in enumerate(mfccs)
    })
    return features

# Función para extraer características lingüísticas
def extraer_linguisticas(transcripcion):
    # Procesamiento de texto con spaCy
    doc = nlp(transcripcion)
    num_palabras = len(doc)
    num_palabras_unicas = len(set([token.text for token in doc]))
    diversidad_lexica = num_palabras_unicas / num_palabras if num_palabras > 0 else 0

    # Embeddings de BERT
    inputs = tokenizer(transcripcion, return_tensors="pt", truncation=True, padding=True)
    outputs = model(**inputs)
    bert_sent_embedding = outputs.last_hidden_state.mean(dim=1).detach().numpy().flatten()

    return {
        "num_palabras": num_palabras,
        "num_palabras_unicas": num_palabras_unicas,
        "diversidad_lexica": diversidad_lexica,
        "bert_embedding": bert_sent_embedding
    }

# Función principal para extraer TODAS las métricas para un chunk
def extraer_metricas_chunk(ruta_audio, transcripcion):
    try:
        # Cargar audio
        y, sr = librosa.load(ruta_audio, sr=None)

        # Características acústicas con librosa (MFCCs)
        mfcc_features = extraer_mfcc(y, sr)

        # Características acústicas con openSMILE
        egemaps_features = smile_egemaps.process_file(ruta_audio).to_dict(orient='records')[0]

        # Características lingüísticas
        linguistics = extraer_linguisticas(transcripcion)

        # Fusionar todas las métricas
        all_features = {
            **mfcc_features,
            **egemaps_features,
            **linguistics  # Incluye características lingüísticas
        }
        return all_features
    except Exception as e:
        print(f"Error procesando el audio {ruta_audio}: {e}")
        return None

In [ ]:
df = pd.read_csv(path_data + 'df_transcrip_chunk_info.csv', encoding='utf-8')
df.head()

In [ ]:
df = pd.read_csv(path_data + 'df_transcrip_chunk_info.csv', encoding='utf-8')

resultados = []

# Procesar cada fila del dataset
for idx, row in df.iterrows():
    ruta_audio = row['name_chunk_audio_path']
    transcripcion = row['Marca']  # Columna con las transcripciones

    # Extraer métricas para cada chunk
    metricas = extraer_metricas_chunk(ruta_audio, transcripcion)

    if metricas:
        # Serializar bert_embedding si está presente
        if 'bert_embedding' in metricas:
            # Convertir ndarray a lista antes de serializar
            if isinstance(metricas['bert_embedding'], np.ndarray):
                metricas['bert_embedding'] = metricas['bert_embedding'].tolist()
            metricas['bert_embedding'] = json.dumps(metricas['bert_embedding'])

        resultado = row.to_dict()  # Copia todas las columnas originales
        resultado.update(metricas)  # Añade las nuevas métricas
        resultados.append(resultado)

# Convertir resultados a DataFrame
df_resultados = pd.DataFrame(resultados)

In [ ]:
df_resultados.head(1)

In [ ]:
print(list(df_resultados.columns))

In [ ]:
df_resultados.to_csv(path_data + 'df_transcrip_audio_metrics.csv', index=False, encoding='utf-8')

# Aphasiabank

In [ ]:
df_aphbank = pd.read_csv(path_data + 'df_aphbank_transcrip_chunk_info.csv')
df_aphbank['name_chunk_audio_path'] = df_aphbank['name_chunk_audio_path'].str.replace(
    '/aphasiabank_es/audios_aphasiabank/',
    '/aphasiabank_en/',
    regex=False
)

df_aphbank.columns

In [ ]:
pd.set_option('display.max_colwidth', None)
df_aphbank[df_aphbank['name_chunk_audio_path'].str.contains('wright207a', case=False, na=False)].head(1)

In [ ]:
pd.set_option('display.max_colwidth', None)
df_aphbank[df_aphbank['name_chunk_audio_path'].str.contains('TCU10a_1957.577_0.783', case=False, na=False)].head(1)

In [ ]:
df_aphbank[df_aphbank['name_chunk_audio']=="02_008_CAT_Conversacion_98.07192_98.80000.wav"]

In [ ]:
len(df_aphbank)

In [ ]:
import numpy as np
import pandas as pd
import librosa
import json
import spacy
from transformers import BertTokenizer, BertModel
# Supongamos que ya tienes importada la librería openSMILE y sus clases correspondientes:
# from opensmile import Smile, FeatureSet, FeatureLevel

# Inicialización de modelos y librerías
nlp = spacy.load("es_core_news_sm")  # Para métricas lingüísticas
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")  # Para embeddings de BERT
model = BertModel.from_pretrained("bert-base-uncased")

# Inicializar OpenSMILE con el feature set eGeMAPSv02
smile_egemaps = Smile(feature_set=FeatureSet.eGeMAPSv02, feature_level=FeatureLevel.Functionals)

def extraer_mfcc(y, sr):
    """
    Extrae MFCCs y calcula estadísticas (media y desviación estándar) para cada coeficiente.
    """
    mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
    features = {f"mfcc{i+1}_mean": np.mean(mfcc) for i, mfcc in enumerate(mfccs)}
    features.update({f"mfcc{i+1}_stddev": np.std(mfcc) for i, mfcc in enumerate(mfccs)})
    return features

def extraer_linguisticas(transcripcion):
    """
    Extrae métricas lingüísticas usando spaCy y BERT.
    """
    doc = nlp(transcripcion)
    num_palabras = len(doc)
    num_palabras_unicas = len(set([token.text for token in doc]))
    diversidad_lexica = num_palabras_unicas / num_palabras if num_palabras > 0 else 0

    # Obtener embedding de BERT para la transcripción
    inputs = tokenizer(transcripcion, return_tensors="pt", truncation=True, padding=True)
    outputs = model(**inputs)
    bert_embedding = outputs.last_hidden_state.mean(dim=1).detach().numpy().flatten()

    return {
        "num_palabras": num_palabras,
        "num_palabras_unicas": num_palabras_unicas,
        "diversidad_lexica": diversidad_lexica,
        "bert_embedding": bert_embedding
    }

def extraer_metricas_chunk(ruta_audio, transcripcion):
    """
    Función que extrae todas las métricas:
      - Estadísticas de MFCCs (media y desviación estándar)
      - Métricas acústicas básicas (duración, amplitud)
      - MFCCs promediados con suavizado (para tener un vector fijo)
      - Características extraídas con openSMILE
      - Características lingüísticas (incluyendo embedding de BERT)
    """
    try:
        # Cargar el audio
        y, sr = librosa.load(ruta_audio, sr=None)
        if len(y) == 0:
            print(f"El audio {ruta_audio} está vacío.")
            return None

        # Ajustar parámetros dinámicamente para la extracción de MFCCs
        n_fft = 2048 if len(y) >= 2048 else len(y)
        hop_length = n_fft // 4 if n_fft >= 4 else 1

        # Extraer MFCCs con parámetros ajustados
        mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13, n_fft=n_fft, hop_length=hop_length)
        if mfccs.shape[1] == 0:
            print(f"No se pudieron extraer MFCCs de {ruta_audio}.")
            return None

        # Suavizado de cada coeficiente con media móvil (parámetro width dinámico)
        n_frames = mfccs.shape[1]
        width = min(9, n_frames)
        mfccs_avg = []
        for i in range(mfccs.shape[0]):
            coeff = mfccs[i]
            avg_coeff = np.convolve(coeff, np.ones(width) / width, mode='valid')
            mfccs_avg.append(avg_coeff)
        mfccs_avg = np.array(mfccs_avg)
        mfccs_promedio = np.mean(mfccs_avg, axis=1)

        # Extraer estadísticas de MFCCs (media y std) usando la función definida
        mfcc_features = extraer_mfcc(y, sr)

        # Extraer características acústicas con openSMILE
        egemaps_features = smile_egemaps.process_file(ruta_audio).to_dict(orient='records')[0]

        # Extraer características lingüísticas
        linguistics = extraer_linguisticas(transcripcion)

        # Calcular otras métricas básicas del audio
        duracion = len(y) / sr
        amplitud_maxima = np.max(y)
        amplitud_minima = np.min(y)
        amplitud_media = np.mean(y)

        # Fusionar todas las métricas en un solo diccionario
        all_features = {
            **mfcc_features,               # mfcc1_mean, mfcc1_stddev, etc.
            **egemaps_features,             # características de openSMILE
            **linguistics,                 # num_palabras, diversidad_lexica, bert_embedding, etc.
            "duracion": duracion,
            "amplitud_maxima": amplitud_maxima,
            "amplitud_minima": amplitud_minima,
            "amplitud_media": amplitud_media,
            "mfccs_promedio": mfccs_promedio.tolist()
        }
        return all_features
    except Exception as e:
        print(f"Error procesando el audio {ruta_audio}: {e}")
        return None

# Lista para almacenar los resultados y archivos con error
resultados_aphbank = []
errores = []

# Contadores
total_archivos = len(df_aphbank)
procesados_ok = 0

# Procesar cada fila del DataFrame
for idx, row in df_aphbank.iterrows():
    ruta_audio = row['name_chunk_audio_path']
    transcripcion = row['Marca']  # Se asume que esta columna contiene la transcripción

    metricas = extraer_metricas_chunk(ruta_audio, transcripcion)

    if metricas:
        # Serializar el embedding de BERT para almacenarlo en formato JSON
        if 'bert_embedding' in metricas:
            if isinstance(metricas['bert_embedding'], np.ndarray):
                metricas['bert_embedding'] = metricas['bert_embedding'].tolist()
            metricas['bert_embedding'] = json.dumps(metricas['bert_embedding'])

        # Combinar todas las columnas originales con las nuevas métricas
        resultado = row.to_dict()  # Copia todas las columnas originales del dataset
        resultado.update(metricas)  # Añade las nuevas métricas extraídas
        resultados_aphbank.append(resultado)
        procesados_ok += 1
    else:
        errores.append(ruta_audio)

# Convertir la lista de resultados en un DataFrame para análisis posteriores
df_aphbank_resultados = pd.DataFrame(resultados_aphbank)

# Mostrar resumen de procesamiento
print("Resumen de procesamiento:")
print(f"Total de archivos en el DataFrame: {total_archivos}")
print(f"Archivos procesados correctamente: {procesados_ok}")
print(f"Archivos con error o sin procesamiento: {len(errores)}")
if errores:
    print("Lista de archivos con error:")
    for archivo in errores:
        print(archivo)

In [ ]:
df_aphbank_resultados

In [ ]:
df_aphbank_resultados.columns

In [ ]:
len(df_aphbank_resultados)